# 🩸 Anemia Prediction using Machine Learning + Explainable AI
**Mini Project | 2025-26**

---
### Notebook Structure
1. Install & Import Libraries
2. Load & Explore Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Model Training (Random Forest, SVM, XGBoost)
6. Model Evaluation & Comparison
7. Explainable AI — SHAP
8. Explainable AI — LIME
9. Conclusion

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Run this cell first to install all required packages
!pip install pandas numpy scikit-learn xgboost shap lime matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# Explainable AI
import shap
import lime
import lime.lime_tabular

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load & Explore Dataset

In [ ]:
# -------------------------------------------------------
# 👉 Upload your CSV file to Replit and set the filename
# -------------------------------------------------------
DATASET_PATH = 'anemia.csv'  # <-- Change this to your uploaded CSV filename

df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

print('\n=== Target Column Value Counts ===')
# Auto-detect target column (last column or named 'Result'/'Anemia'/'label')
TARGET_COL = df.columns[-1]
print(f'Target column detected: "{TARGET_COL}"')
print(df[TARGET_COL].value_counts())

## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[TARGET_COL].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution (Bar)', fontsize=13, fontweight='bold')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

df[TARGET_COL].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                    colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Class Distribution (Pie)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of numerical features
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)

n = len(numeric_cols)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='#3498db', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots by target class
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by=TARGET_COL, ax=axes[i],
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col} by {TARGET_COL}', fontweight='bold')
    axes[i].set_xlabel(TARGET_COL)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature vs Target (Boxplots)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚙️ Step 4: Data Preprocessing

In [ ]:
df_clean = df.copy()

# Handle missing values
for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Encode categorical columns
le = LabelEncoder()
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f'Encoded: {col}')

# Separate features and target
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

# Encode target if needed
if y.dtype == 'object':
    y = le.fit_transform(y)

FEATURE_NAMES = X.columns.tolist()
print(f'\n✅ Features: {FEATURE_NAMES}')
print(f'✅ Target classes: {np.unique(y)}')
print(f'✅ Dataset shape: X={X.shape}, y={y.shape}')

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set : {X_train.shape}')
print(f'Testing set  : {X_test.shape}')
print('✅ Scaling done!')

## 🤖 Step 5: Model Training

In [ ]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print('✅ Random Forest trained!')

In [ ]:
# --- SVM ---
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
print('✅ SVM trained!')

In [ ]:
# --- XGBoost ---
xgb_model = XGBClassifier(n_estimators=100, random_state=42,
                           use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
print('✅ XGBoost trained!')

## 📈 Step 6: Model Evaluation & Comparison

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    acc  = accuracy_score(y_te, y_pred)
    auc  = roc_auc_score(y_te, y_prob)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()
    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(f'  Accuracy      : {acc:.4f}')
    print(f'  AUC-ROC       : {auc:.4f}')
    print(f'  CV Accuracy   : {cv:.4f}')
    print(f'\n{classification_report(y_te, y_pred)}')
    return {'Model': name, 'Accuracy': acc, 'AUC-ROC': auc, 'CV Accuracy': cv,
            'y_pred': y_pred, 'y_prob': y_prob}

rf_res  = evaluate_model('Random Forest', rf_model,  X_train, X_test, y_train, y_test)
svm_res = evaluate_model('SVM',           svm_model, X_train_scaled, X_test_scaled, y_train, y_test)
xgb_res = evaluate_model('XGBoost',       xgb_model, X_train, X_test, y_train, y_test)

In [ ]:
# Model Comparison Bar Chart
results_df = pd.DataFrame([
    {'Model': rf_res['Model'],  'Accuracy': rf_res['Accuracy'],  'AUC-ROC': rf_res['AUC-ROC']},
    {'Model': svm_res['Model'], 'Accuracy': svm_res['Accuracy'], 'AUC-ROC': svm_res['AUC-ROC']},
    {'Model': xgb_res['Model'], 'Accuracy': xgb_res['Accuracy'], 'AUC-ROC': xgb_res['AUC-ROC']},
])

results_df.set_index('Model')[['Accuracy', 'AUC-ROC']].plot(
    kind='bar', figsize=(9, 5), color=['#3498db', '#e74c3c'],
    edgecolor='black', width=0.5
)
plt.title('Model Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0.5, 1.05)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(results_df.to_string(index=False))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, res in zip(axes, [rf_res, svm_res, xgb_res]):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{res['Model']}", fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(8, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71']
for res, color in zip([rf_res, svm_res, xgb_res], colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{res['Model']} (AUC={res['AUC-ROC']:.3f})", color=color, lw=2)
plt.plot([0,1],[0,1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Step 7: Explainable AI — SHAP
> SHAP (SHapley Additive exPlanations) shows **how much each feature contributes** to a prediction globally and per-sample.

In [ ]:
# SHAP with XGBoost (best model)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Global Feature Importance — Summary Plot
plt.figure()
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Summary Plot — Global Feature Importance', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 Features pushed RIGHT (positive SHAP) increase anemia probability.')
print('📌 Features pushed LEFT (negative SHAP) decrease anemia probability.')

In [ ]:
# SHAP Bar Plot — Mean absolute importance
plt.figure()
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES,
                  plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP|)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Local Explanation — Single Prediction
SAMPLE_IDX = 0  # Change this to explain a different test sample

print(f'Explaining prediction for Test Sample #{SAMPLE_IDX}')
print(f'True Label   : {y_test.iloc[SAMPLE_IDX] if hasattr(y_test, "iloc") else y_test[SAMPLE_IDX]}')
print(f'Predicted    : {xgb_model.predict(X_test.iloc[[SAMPLE_IDX]])[0]}')

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[SAMPLE_IDX],
    X_test.iloc[SAMPLE_IDX],
    feature_names=FEATURE_NAMES,
    matplotlib=True,
    show=False
)
plt.title(f'SHAP Force Plot — Sample {SAMPLE_IDX}', fontweight='bold', pad=40)
plt.savefig('shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Waterfall Plot
shap_explanation = shap.Explanation(
    values=shap_values[SAMPLE_IDX],
    base_values=explainer.expected_value,
    data=X_test.iloc[SAMPLE_IDX].values,
    feature_names=FEATURE_NAMES
)
plt.figure()
shap.waterfall_plot(shap_explanation, show=False)
plt.title(f'SHAP Waterfall — Sample {SAMPLE_IDX}', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔬 Step 8: Explainable AI — LIME
> LIME (Local Interpretable Model-agnostic Explanations) explains **individual predictions** by fitting a simple local model around that data point.

In [ ]:
# LIME Explainer Setup
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=FEATURE_NAMES,
    class_names=['No Anemia', 'Anemia'],
    mode='classification'
)
print('✅ LIME explainer initialized!')

In [ ]:
# LIME Local Explanation — Sample
LIME_SAMPLE_IDX = 5  # Change to explain a different sample

lime_exp = lime_explainer.explain_instance(
    data_row=np.array(X_test.iloc[LIME_SAMPLE_IDX]),
    predict_fn=xgb_model.predict_proba,
    num_features=len(FEATURE_NAMES)
)

print(f'LIME Explanation for Test Sample #{LIME_SAMPLE_IDX}')
print(f'True Label   : {y_test.iloc[LIME_SAMPLE_IDX] if hasattr(y_test, "iloc") else y_test[LIME_SAMPLE_IDX]}')
print(f'Predicted    : {xgb_model.predict(X_test.iloc[[LIME_SAMPLE_IDX]])[0]}')
print(f'Probabilities: {xgb_model.predict_proba(X_test.iloc[[LIME_SAMPLE_IDX]])[0]}')

fig = lime_exp.as_pyplot_figure()
plt.title(f'LIME Explanation — Sample {LIME_SAMPLE_IDX}', fontweight='bold')
plt.tight_layout()
plt.savefig('lime_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LIME — Compare explanations for 3 different samples
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sample_indices = [0, 5, 10]

for ax, idx in zip(axes, sample_indices):
    exp = lime_explainer.explain_instance(
        data_row=np.array(X_test.iloc[idx]),
        predict_fn=xgb_model.predict_proba,
        num_features=5
    )
    feat_weights = exp.as_list()
    features = [fw[0][:25] for fw in feat_weights]
    weights  = [fw[1] for fw in feat_weights]
    colors   = ['#e74c3c' if w > 0 else '#3498db' for w in weights]
    ax.barh(features, weights, color=colors, edgecolor='black')
    true_label = y_test.iloc[idx] if hasattr(y_test, 'iloc') else y_test[idx]
    ax.set_title(f'Sample {idx} | True: {true_label}', fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Weight')

plt.suptitle('LIME — Local Explanations for 3 Samples\n(Red = pushes toward Anemia | Blue = pushes away)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('lime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Step 9: Conclusion

In [ ]:
print('=' * 60)
print('  ANEMIA PREDICTION — FINAL RESULTS SUMMARY')
print('=' * 60)

for res in [rf_res, svm_res, xgb_res]:
    print(f"  {res['Model']:<20} Accuracy: {res['Accuracy']:.4f}  AUC: {res['AUC-ROC']:.4f}")

best = max([rf_res, svm_res, xgb_res], key=lambda r: r['Accuracy'])
print(f"\n  🏆 Best Model : {best['Model']} with Accuracy = {best['Accuracy']:.4f}")

print('''
CONCLUSIONS:
  • Machine learning successfully predicts anemia from blood parameters.
  • XGBoost achieved the highest accuracy and AUC-ROC score.
  • SHAP identified the most globally influential diagnostic features.
  • LIME provided patient-level explanations for individual predictions.
  • The system is transparent, interpretable, and suitable for clinical use.
''')

print('Files saved:')
import os
for f in os.listdir('.'):
    if f.endswith('.png'):
        print(f'  📊 {f}')